In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg21zn25_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 276 atoms, Mg126Zn150


In [2]:
slab = surface(alloy, (2, 0, 1), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(2,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need {25/21:.4f})")
print(f"Stoichiometric: {'YES' if zn*21 == mg*25 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 552, Mg: 252, Zn: 300
Thickness: 13.9 A
Ratio Zn/Mg: 1.1905 (need 1.1905)
Stoichiometric: YES
Symmetric: False


In [3]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Using tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Using tolerance: 0.020 A

Total atomic layers: 241

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0     Mg2Zn1     0.013  |      240        Mg2     0.005 NO <---
      1     Mg1Zn1     0.077  |      239     Mg1Zn1     0.062    YES
      2     Mg1Zn2     0.194  |      238     Mg1Zn2     0.180    YES
      3     Mg2Zn3     0.245  |      237     Mg2Zn3     0.230    YES
      4     Mg1Zn1     0.332  |      236     Mg1Zn1     0.318    YES
      5        Mg1     0.377  |      235        Mg1     0.362    YES
      6        Zn1     0.422  |      234        Zn1     0.407    YES
      7     Mg1Zn1     0.445  |      233     Mg1Zn1     0.430    YES
      8        Zn2     0.483  |      232        Zn2     0.469    YES
      9        Mg1     0.558  |      231        Mg1     0.544    YES
     10     Mg1Zn2     0.578  |      230     Mg1Zn2     0.564    YES
     11        Mg1     0.598  |      2

In [4]:
# Search for symmetric removals
print("Searching for symmetric removals...\n")

for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(2,0,1), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
        except:
            continue

Searching for symmetric removals...

  bot_rm=1, top_rm=1: 547 atoms, Mg248 Zn299, removed 3+2=5
  bot_rm=2, top_rm=2: 543 atoms, Mg246 Zn297, removed 5+4=9
  bot_rm=3, top_rm=3: 537 atoms, Mg244 Zn293, removed 8+7=15
  bot_rm=4, top_rm=4: 527 atoms, Mg240 Zn287, removed 13+12=25
  bot_rm=5, top_rm=5: 523 atoms, Mg238 Zn285, removed 15+14=29
  bot_rm=6, top_rm=6: 521 atoms, Mg236 Zn285, removed 16+15=31
  bot_rm=7, top_rm=7: 519 atoms, Mg236 Zn283, removed 17+16=33


In [5]:
removed_bot = layer_idx[0]
removed_top = layer_idx[n-1]
removed_all = removed_bot + removed_top

keep = []
for j in range(1, n - 1):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_all)
rem_zn = sum(sym[j] == 'Zn' for j in removed_all)
print(f"\nRemoved: {len(removed_all)} atoms (Mg{rem_mg} Zn{rem_zn})")

print(f"\nBottom ({len(removed_bot)}):")
for j in removed_bot:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

print(f"\nTop ({len(removed_top)}):")
for j in removed_top:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab)
keep_set = set(keep)

paired = []
unpaired = []
for j in removed_all:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    matched = False
    for k in removed_all:
        if k == j:
            continue
        if np.linalg.norm(slab.positions[k] - inv_cart) < 0.5:
            paired.append((j, k))
            matched = True
            break
    if not matched:
        unpaired.append(j)

print(f"\nPaired within removed: {len(paired)//2} pairs")
print(f"Unpaired: {len(unpaired)}")
for j in unpaired:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) "
          f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

SG = P-1
Inversion: f -> [0.62357808 0.81178904 0.        ] - f

Removed: 5 atoms (Mg4 Zn1)

Bottom (3):
  idx=2 Zn (31.484, 0.000, 8.000)
  idx=95 Mg (5.665, 18.610, 8.015)
  idx=68 Mg (9.680, 10.482, 8.025)

Top (2):
  idx=483 Mg (1.156, 8.812, 21.854)
  idx=457 Mg (5.171, 0.684, 21.864)

Paired within removed: 2 pairs
Unpaired: 1
  idx=2 Zn (31.484, 0.000, 8.000) -> inv: (10.836, 19.294, 21.864)


In [6]:
# 1 orphan Zn — need supercell to double to 2. 1x2 along y.

slab_2y = make_supercell(slab, [[1,0,0],[0,2,0],[0,0,1]])

sym2 = np.array(slab_2y.get_chemical_symbols())
mg2, zn2 = sum(sym2 == 'Mg'), sum(sym2 == 'Zn')
z2 = slab_2y.get_positions()[:, 2]
order2 = np.argsort(z2)
print(f"1x2 supercell: {len(slab_2y)} atoms, Mg{mg2} Zn{zn2}")
print(f"Stoichiometric: {'YES' if zn2*21 == mg2*25 else 'NO'}")

z_sorted2 = np.sort(z2)
gaps2 = np.diff(z_sorted2)
tol2 = max(0.02, min(0.3, np.median(gaps2[gaps2 > 0.01]) / 2))

layers2 = []
layer_idx2 = []
cur = [order2[0]]
for i in range(1, len(order2)):
    if z2[order2[i]] - z2[order2[i-1]] < tol2:
        cur.append(order2[i])
    else:
        layers2.append(Counter(sym2[j] for j in cur))
        layer_idx2.append(list(cur))
        cur = [order2[i]]
layers2.append(Counter(sym2[j] for j in cur))
layer_idx2.append(list(cur))
n2 = len(layers2)

# Remove 1 from each end
removed_bot2 = layer_idx2[0]
removed_top2 = layer_idx2[n2-1]
keep2 = []
for j in range(1, n2 - 1):
    keep2.extend(layer_idx2[j])

trimmed2 = slab_2y[keep2]
ps_trim2 = AseAtomsAdaptor().get_structure(trimmed2)

slab_trim2 = Slab(ps_trim2.lattice, ps_trim2.species, ps_trim2.frac_coords,
    miller_index=(2,0,1), oriented_unit_cell=ps_trim2, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"After removing 1 from each end: {len(trimmed2)} atoms, Symmetric = {slab_trim2.is_symmetric()}")

if slab_trim2.is_symmetric():
    sga2 = SpacegroupAnalyzer(ps_trim2, symprec=0.1)
    print(f"SG = {sga2.get_space_group_symbol()}")
    for op in sga2.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break

    # Find removed and unpaired
    removed_all2 = removed_bot2 + removed_top2
    print(f"\nRemoved: {len(removed_all2)} atoms")
    
    ps_orig2 = AseAtomsAdaptor().get_structure(slab_2y)
    keep_set2 = set(keep2)
    
    # Find bot-top pairs
    bot_rem = removed_bot2
    top_rem = removed_top2
    paired = []
    for j in bot_rem:
        frac = ps_orig2[j].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
        for k in top_rem:
            if np.linalg.norm(slab_2y.positions[k] - inv_cart) < 0.5:
                paired.append((j, k))
                break
    
    used_top = set(k for _, k in paired)
    unpaired_bot = [j for j in bot_rem if j not in [b for b, _ in paired]]
    unpaired_top = [j for j in top_rem if j not in used_top]
    
    print(f"Paired: {len(paired)}, Unpaired bot: {len(unpaired_bot)}, Unpaired top: {len(unpaired_top)}")
    
    for j in unpaired_bot:
        frac = ps_orig2[j].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
        pos = slab_2y.positions[j]
        print(f"  idx={j} {sym2[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) "
              f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

1x2 supercell: 1104 atoms, Mg504 Zn600
Stoichiometric: YES
After removing 1 from each end: 1094 atoms, Symmetric = True
SG = P-1
Inversion: f -> [0.62357808 0.90589452 0.        ] - f

Removed: 10 atoms
Paired: 4, Unpaired bot: 2, Unpaired top: 0
  idx=554 Zn (-10.836, 23.767, 8.000) -> inv: (10.836, 19.294, 21.864)
  idx=2 Zn (-0.000, 0.000, 8.000) -> inv: (0.000, 43.061, 21.864)


In [7]:
sc_fixed = slab_2y.copy()
ps_orig2 = AseAtomsAdaptor().get_structure(slab_2y)

# Keep idx=2 on bottom, move idx=554 to inv(2)
keep_idx = 2
move_idx = 554

frac = ps_orig2[keep_idx].frac_coords
inv_frac = (inv_translation - frac) % 1.0
inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)

old = sc_fixed.positions[move_idx].copy()
sc_fixed.positions[move_idx] = inv_cart
print(f"Moved {move_idx} -> inv({keep_idx}): "
      f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
      f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(2,0,1), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Moved 554 -> inv(2): (-10.836, 23.767, 8.000) -> (0.000, 43.061, 21.864)

Atoms: 1104, Mg: 504, Zn: 600
Stoichiometric: YES
Symmetric: True


In [8]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [9]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_201_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_201_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg21zn25_201_2layers_reconstructed_ase.data
  Atoms: 1104
  Thickness: 13.9 A
  Area: 1496.6 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [10]:
#unrelaxed surface energy calculation
E_slab =   -1328.6717    # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 1104 / 46                # formula units in slab =
A = 1496.6             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -1453.88360 eV
E_slab - n*E_bulk = 125.21190 eV
S = 0.041832 eV/Ang^2
S = 0.6702 J/m^2


In [12]:
#relaxed surface energy calculation
E_slab_relaxed = -1338.34701845946  # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 1104 / 46                      # 32 formula units
A = 1496.6                  # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6702
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 115.53658 eV
S relaxed = 0.038600 eV/Ang^2
S relaxed = 0.6184 J/m^2

Unrelaxed S = 0.6702 J/m^2
Relaxed S   = 0.6184 J/m^2
Relaxation energy = 0.0518 J/m^2
